Load Pandas, plotting and [Sklearn](https://scikit-learn.org/stable/) packages.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn import metrics
from sklearn.naive_bayes import MultinomialNB
from sklearn import preprocessing

## Naive Bayes by hand


### Copy the data from the lecture example

In [ ]:
Weather
Play
Rainy
No
Sunny
Yes
Overcast
Yes
Overcast
No
Sunny
Yes
Rainy
No
Rainy
No
Sunny
Yes
Overcast
Yes
Overcast
Yes
Rainy
No
Rainy
Yes


In [2]:
mixed = """
Weather
Play
Rainy
No
Sunny
Yes
Overcast
Yes
Overcast
No
Sunny
Yes
Rainy
No
Rainy
No
Sunny
Yes
Overcast
Yes
Overcast
Yes
Rainy
No
Rainy
Yes
""".split()
mixed


['Weather',
 'Play',
 'Rainy',
 'No',
 'Sunny',
 'Yes',
 'Overcast',
 'Yes',
 'Overcast',
 'No',
 'Sunny',
 'Yes',
 'Rainy',
 'No',
 'Rainy',
 'No',
 'Sunny',
 'Yes',
 'Overcast',
 'Yes',
 'Overcast',
 'Yes',
 'Rainy',
 'No',
 'Rainy',
 'Yes']

In [3]:
weather = mixed[2::2]
play = mixed[3::2]
print( weather )
print( play )

['Rainy', 'Sunny', 'Overcast', 'Overcast', 'Sunny', 'Rainy', 'Rainy', 'Sunny', 'Overcast', 'Overcast', 'Rainy', 'Rainy']
['No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes']


In [4]:
pd.Series(play).value_counts(dropna=False)

,count
Yes,7
No,5


### Build a data frame starting from an empty data frame

In [5]:
# Create empty data frame
tennis_df = pd.DataFrame()
tennis_df

""


In [6]:
# Append Weather column
tennis_df["Weather"] = weather
tennis_df

,Weather
0,Rainy
1,Sunny
2,Overcast
3,Overcast
4,Sunny
5,Rainy
6,Rainy
7,Sunny
8,Overcast
9,Overcast


In [7]:
# Append Play column
tennis_df["Play"] = play
tennis_df

,Weather,Play
0,Rainy,No
1,Sunny,Yes
2,Overcast,Yes
3,Overcast,No
4,Sunny,Yes
5,Rainy,No
6,Rainy,No
7,Sunny,Yes
8,Overcast,Yes
9,Overcast,Yes


In [8]:
tennis_df.shape

(12, 2)

### Weather probabilities

In [9]:
weather_counts = tennis_df["Weather"].value_counts()
weather_counts

,count
Weather,
Rainy,5
Overcast,4
Sunny,3


In [10]:
# weather_total = len(tennis_df["Weather"])
weather_total = weather_counts.sum()
weather_total


np.int64(12)

In [11]:
p_weather = weather_counts / weather_total
p_weather

,count
Weather,
Rainy,0.416667
Overcast,0.333333
Sunny,0.250000


In [12]:
tennis_df["Weather"].value_counts(normalize=True)*100

,proportion
Weather,
Rainy,41.666667
Overcast,33.333333
Sunny,25.000000


### Play probabilities

In [13]:
play_counts = tennis_df["Play"].value_counts().sort_index()
play_counts

,count
Play,
No,5
Yes,7


In [14]:
play_total = play_counts.sum()
play_total

np.int64(12)

In [15]:
p_play = play_counts/play_total
p_play

,count
Play,
No,0.416667
Yes,0.583333


In [16]:
tennis_df["Play"].value_counts(normalize=True).sort_index() * 100

,proportion
Play,
No,41.666667
Yes,58.333333


### Conditional probabilities

In [28]:
tennis_df.columns.to_list()


['Weather', 'Play']

In [29]:
cond_counts = tennis_df.value_counts()
cond_counts

Weather   Play
Rainy     No      4
Sunny     Yes     3
Overcast  Yes     3
          No      1
Rainy     Yes     1
Name: count, dtype: int64

In [30]:
type(cond_counts)

pandas.core.series.Series

In [32]:
cond_counts["Overcast",:]

,count
Play,
Yes,3
No,1


In [33]:
cond_counts[:,"Yes"]

,count
Weather,
Sunny,3
Overcast,3
Rainy,1


In [34]:
cond_counts["Overcast","Yes"]

np.int64(3)

### Bayes theorem

$ P(\text{Yes} \mid \text{Overcast}) = \frac{P(\text{Overcast} \mid \text{Yes}) \cdot P(\text{Yes})}{P(\text{Overcast})} $


In [35]:
p_overcast_yes = cond_counts["Overcast", "Yes"]/play_counts["Yes"]
p_overcast_yes

np.float64(0.42857142857142855)

In [36]:
p_yes = p_play["Yes"]
p_yes

np.float64(0.5833333333333334)

In [37]:
p_overcast = p_weather["Overcast"]
p_overcast

np.float64(0.3333333333333333)

In [38]:
p_yes_overcast = p_overcast_yes * p_yes / p_overcast
p_yes_overcast

np.float64(0.75)

## Naive Bayes

In [39]:
# Separarate target and features
target = tennis_df["Play"]
features = tennis_df.drop( columns = "Play" )


In [40]:
target, features

(0      No
 1     Yes
 2     Yes
 3      No
 4     Yes
 5      No
 6      No
 7     Yes
 8     Yes
 9     Yes
 10     No
 11    Yes
 Name: Play, dtype: object,
      Weather
 0      Rainy
 1      Sunny
 2   Overcast
 3   Overcast
 4      Sunny
 5      Rainy
 6      Rainy
 7      Sunny
 8   Overcast
 9   Overcast
 10     Rainy
 11     Rainy)

In [41]:
# Create labelEncoder
le = preprocessing.LabelEncoder()

# Convert string labels into numbers. Note: this isn't the best way to encode
# weather but we will talk about other encoding methods later on
weather_encoded = le.fit_transform(features["Weather"])
print('Weather:', weather_encoded)


Weather: [1 2 0 0 2 1 1 2 0 0 1 1]


In [42]:
label_encoded = le.fit_transform(target)
print('Label:', label_encoded)

Label: [0 1 1 0 1 0 0 1 1 1 0 1]


In [43]:
dict(zip(label_encoded,target))

{np.int64(0): 'No', np.int64(1): 'Yes'}

In [44]:
dict(zip(weather_encoded,features["Weather"]))

{np.int64(1): 'Rainy', np.int64(2): 'Sunny', np.int64(0): 'Overcast'}

In [45]:
#Create a Multinomial Naive Bayes Classifier
model = MultinomialNB()

# Train the model using the training sets
model.fit(weather_encoded.reshape(-1,1), label_encoded)

#Predict Output
predicted = model.predict( [[0]] )     # 0: Overcast
print( "Predicted Value:", ("No","Yes")[predicted[0]] ) # 0: No    1: Yes


Predicted Value: Yes


In [46]:
model.predict_proba([[0]])[0]*100

array([41.66666667, 58.33333333])

In [47]:
weather_encoded, weather_encoded.reshape(-1,1)

(array([1, 2, 0, 0, 2, 1, 1, 2, 0, 0, 1, 1]),
 array([[1],
        [2],
        [0],
        [0],
        [2],
        [1],
        [1],
        [2],
        [0],
        [0],
        [1],
        [1]]))

In [48]:
weather_encoded.reshape(-1,1).shape

(12, 1)

### Integer and Floating point precision ( side bar )


In [49]:
big_int = 123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789000000
big_int


123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789000000

In [50]:
len(str(big_int))


105

In [51]:
big_int + 1


123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789000001

In [52]:
float(big_int)


1.2345678901234568e+104

In [53]:
float(big_int)+1


1.2345678901234568e+104

In [54]:
int(float(big_int))-big_int


-2594971299523355466297607200432343922460668439393063880379049059784787282748375667224384

In [55]:
big_int-(big_int+1)


-1

In [56]:
type(big_int)


int